In [2]:
import pickle
import sys
import os

# Add parent directory to path for imports
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))
from utils.graph_utils import (build_knn_graph_from_features, save_graph, set_random_seeds, load_graph, 
                              get_device, compute_auxiliary_features, compute_topological_features,
                              prepare_pytorch_geometric_data)

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Build Grap from Image Dataset
+ In the built Graph:--> homogeneous graph
+ nodes: node ID are numbers; attributes have {x:[512, ], y: np.int64(class_label), train: bool, val: bool, test: bool}
    + x = shape(512,) is the output of ResNet model, aka. a one dimention vector of learned image features;

+ edges: (head_id, tail_id, edge_weight)
    + each head_id has at most k-1 connected tail nodes decided by k-nearest neighbors;
    + no self edge.
    + edge_weight is the similarity score between two nodes (two images) calculated by the cosine similarity of features(x);
    + edges can also exist between nodes with the same labels;
    + rewiring edegs for better homophily?: remove edges if similarity score less than 0.65 and the nodes belong to different classes;


In [2]:
G = load_graph("../datasets/G_Bloodmnist_inductive.gpickle")
for node, attrs in G.nodes(data=True):
    print(attrs)
    break

Loaded graph from ../datasets/G_Bloodmnist_inductive.gpickle: 17092 nodes, 60116 edges
{'x': array([4.05756569e+00, 2.05689985e-02, 2.14794874e-01, 1.22916043e+00,
       1.19370139e+00, 1.07704028e-02, 4.48740873e-04, 3.61270201e-03,
       2.00523806e+00, 5.50419688e-01, 2.49505699e-01, 1.08972228e+00,
       5.99166930e-01, 5.33580601e-01, 7.22724974e-01, 2.21812353e-01,
       3.32862198e-01, 1.51974440e+00, 1.78651094e-01, 1.17551482e+00,
       1.67127073e+00, 7.67934084e-01, 9.82250366e-03, 0.00000000e+00,
       6.36812866e-01, 2.27199979e-02, 1.68052828e+00, 1.63775384e+00,
       1.12941384e-01, 1.29523671e+00, 5.51270306e-01, 1.61917782e+00,
       2.08937854e-01, 1.35526687e-01, 1.19500928e-01, 6.11274302e-01,
       0.00000000e+00, 7.80010223e-03, 2.17689872e+00, 2.40447819e-02,
       5.83271645e-02, 1.19344182e-01, 5.90196289e-02, 9.19623613e-01,
       1.09254003e+00, 1.23408802e-01, 8.61722350e-01, 5.46605349e-01,
       5.66034913e-01, 2.20643544e+00, 4.31099087e-01, 

In [3]:
i = 0
for edge in G.edges(data=True):
    print(edge)
    i += 1
    if i >20:
        break

(0, np.int64(2358), {'weight': np.float64(0.9460398554801941)})
(0, np.int64(13343), {'weight': np.float64(0.945427417755127)})
(0, np.int64(14983), {'weight': np.float64(0.9448888897895813)})
(0, np.int64(7597), {'weight': np.float64(0.9447507858276367)})
(0, np.int64(5958), {'weight': np.float64(0.9439164400100708)})
(0, np.int64(2991), {'weight': np.float64(0.9427925944328308)})
(0, np.int64(3436), {'weight': np.float64(0.9425932168960571)})
(1, np.int64(10636), {'weight': np.float64(0.9510995149612427)})
(1, np.int64(2743), {'weight': np.float64(0.9457296133041382)})
(1, np.int64(9622), {'weight': np.float64(0.9442079067230225)})
(1, np.int64(15634), {'weight': np.float64(0.9384658336639404)})
(1, np.int64(10132), {'weight': np.float64(0.9375439286231995)})
(1, np.int64(13104), {'weight': np.float64(0.9367461800575256)})
(1, np.int64(7044), {'weight': np.float64(0.9365302920341492)})
(2, np.int64(2346), {'weight': np.float64(0.934360146522522)})
(2, np.int64(3973), {'weight': np.fl

# pythorch_geometric Data

- data attributes:
    + x=torch.tensor(x, dtype=torch.float),
    + edge_index=edge_index,
    + edge_attr=edge_attr,
    + y=torch.tensor(y, dtype=torch.long),
    + train_mask=torch.tensor(train_mask,dtype=torch.bool),
    + val_mask=torch.tensor(val_mask, dtype=torch.bool),
    + test_mask=torch.tensor(test_mask, dtype=torch.bool),
    + topo_features=torch.tensor(topo_features, dtype=torch.float),
    + topo_scaler_mean=torch.tensor(scaler.mean_, dtype=torch.float),
    + topo_scaler_scale=torch.tensor(scaler.scale_, dtype=torch.float)
    + sim_scores = torch.tensor(auxiliary_features['sim_scores'], dtype=torch.float)
    + data.homophily = torch.tensor(auxiliary_features['homophily'], dtype=torch.float)
    + data.entropies = torch.tensor(auxiliary_features['entropies'], dtype=torch.float)

- Topological features --- calculaed from the Graph
    + 'degrees': 
    + 'clustering':
    + 'two_hop_agreement': 
    + 'eigenvector_centrality':
    + 'degree_centrality':
    + 'avg_edge_weight': 

- auxiliary features
    + homophily,h(v): qunatify how much a node shares same labels with its neighbors;
    + similarity entropy, s(v): quantify the sharpness of each node's neighbors' distribution. It measures the uncertainty in edge-weight distributions over node-v's neighborhood.

## Fuzzy Model Implementation

#### (1) Topological Feature Vector (per node)

For each node \( u \), they define a topological feature vector:

$$
\mathbf{f}_u =
\begin{bmatrix}
d(u),\; C(u),\; L(u)
\end{bmatrix}
\in \mathbb{R}^3
$$

where:
- $ d(u) $: node degree  
- $ C(u) $: clustering coefficient  
- $ L(u) $: 2-hop label agreement  

---

#### (2) Fuzzy Rule Activation (Eq. 3)

Each topological feature corresponds to **one fuzzy rule**.  
The activation of rule $ i $ for node $ u $ is defined as:

$$
r_i(u) = \sigma\big(\alpha_i (f_u[i] - \theta_i)\big)
$$

where:
- $ \sigma(\cdot) $ is the sigmoid function  
- $ \theta_i $ is a **learnable threshold**
- $ \alpha_i $ is a **learnable sharpness parameter**

---

##### Rule Design

- **One rule per feature**
- Learnable parameters:
  - Threshold $ \theta_i $
  - Sharpness $ \alpha_i $

---

##### Rule Vector

Collecting all rule activations yields the rule vector:

$$
\mathbf{r}(u) \in [0,1]^3
$$

This vector provides an interpretable, symbolic description of the node’s structural properties.

---

#### (3) Rule Projection (Eq. 4)

The rule vector is projected into the node embedding space:

$$
\mathbf{e}_u = \mathbf{W}_r \mathbf{r}(u) + \mathbf{b}_r \in \mathbb{R}^d
$$

where:
- $ \mathbf{W}_r \in \mathbb{R}^{d \times 3} $
- $ \mathbf{b}_r \in \mathbb{R}^d $

---

#### (4) Gating Mechanism (Eq. 5)

A gate controls the contribution of neural vs. symbolic information:

$$
\mathbf{g}_u = \sigma\big(\mathbf{W}_g [\mathbf{h}_u \,\|\, \mathbf{e}_u] + \mathbf{b}_g\big)
$$

where:
- $ \mathbf{h}_u \in \mathbb{R}^d $ is the GNN embedding
- $ \| $ denotes concatenation
- $ \mathbf{g}_u \in [0,1]^d $

---

#### (5) Fusion of Neural and Symbolic Representations (Eq. 6)

The final node representation is computed as:

$$
\mathbf{h}'_u = \mathbf{g}_u \odot \mathbf{h}_u + (1 - \mathbf{g}_u) \odot \mathbf{e}_u
$$

where $ \odot $ denotes element-wise multiplication.

This formulation allows the model to **adaptively balance learned embeddings and interpretable fuzzy rules**.


In [4]:
"""
Fuzzy rule-enhanced GNN models for FireGNN framework.
Explicitly follow the paper equations.
Combines GNN architectures with trainable fuzzy rules.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, GINConv
from torch_geometric.data import Data

class PaperFuzzyRuleLayer(nn.Module):
    """
    FireGNN fuzzy rules exactly as defined in the paper.
    Implements Eq. (3): r_i(u) = sigmoid(alpha_i * (f_i(u) - theta_i))
    """

    def __init__(self, num_rules=3):
        super().__init__()
        self.num_rules = num_rules

        # Learnable thresholds θ_i
        self.theta = nn.Parameter(torch.zeros(num_rules))

        # Learnable sharpness α_i (initialized positive)
        self.alpha = nn.Parameter(torch.ones(num_rules))

    def forward(self, topo_features):
        """
        Args:
            topo_features: Tensor [N, 3]
                [degree, clustering coefficient, 2-hop label agreement]

        Returns:
            r: Tensor [N, 3] fuzzy rule activations
        """
        # Ensure correct dimensionality
        assert topo_features.size(1) == self.num_rules

        # r_i(u) = sigmoid(alpha_i * (f_i(u) - theta_i))
        r = torch.sigmoid(self.alpha * (topo_features - self.theta))
        return r
    
class PaperFuzzyGCN(nn.Module):
    """
    FireGNN GCN model exactly matching the paper formulation.
    """

    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_layers=2, dropout=0.5):
        super().__init__()

        self.num_layers = num_layers
        self.dropout = dropout
        d = hidden_channels

        # ---------- GCN backbone ----------
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels))

        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))

        self.bns = nn.ModuleList(
            [nn.BatchNorm1d(hidden_channels) for _ in range(num_layers - 1)]
        )

        # ---------- Fuzzy rules ----------
        self.fuzzy_rules = PaperFuzzyRuleLayer(num_rules=3)

        # Rule projection: Eq. (4) & Gate: Eq. (5)
        self.gate = nn.Linear(num_layers + hidden_channels, hidden_channels)

        # Classifier
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, topo_features):
        """
        Args:
            x: Node features [N, in_channels]
            edge_index: Graph edges
            topo_features: [N, 3] (degree, clustering, 2-hop agreement)
        """

        # ----- GCN forward -----
        for i in range(self.num_layers - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        h = self.convs[-1](x, edge_index)  # h_u

        # ----- Fuzzy rule activation (Eq. 3) -----
        r = self.fuzzy_rules(topo_features)  # [N, 3]

        # ----- Rule projection & Gating (Eq. 5) -----
        g = torch.sigmoid(self.gate(torch.cat([h, r], dim=1)))

        # ----- Fusion (Eq. 6) -----
        h_prime = g * h + (1 - g) * r

        # ----- Classification -----
        out = self.classifier(h_prime)

        return F.log_softmax(out, dim=1), r

### Algorithm 1: FireGNN End-to-End Training Procedure

**Input:**  
Graph $G = (V, E)$, node features $X \in \mathbb{R}^{|V| \times d_{\text{in}}}$,  
node labels $Y$, training node indices $V_L$.

**Parameters:**  
- GNN weights $\{W^{(\ell)}\}_{\ell=1}^{L}$  
- Fuzzy rule parameters $\{\theta_i, \alpha_i\}_{i=1}^{3}$  
- Fusion parameters $\{W_r, b_r, W_g, b_g\}$  

**Output:**  
Trained FireGNN model parameters.

---

#### Procedure: TRAINFIREGNN($ G, X, Y, V_L $)

1. **Initialize** all model parameters.

2. **For each training epoch:**

   ##### GNN Forward Pass
   - Initialize node embeddings:
     $$
     H^{(0)} \leftarrow X
     $$

   - For $\ell = 0$ to $L - 1$:
     $$
     H^{(\ell+1)} \leftarrow \sigma\!\left(
     \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}
     H^{(\ell)} W^{(\ell)}
     \right)
     $$
     where $\tilde{A} = A + I$ and $\tilde{D}$ is the degree matrix of $\tilde{A}$.

   - Final GNN embeddings:
     $$
     H \leftarrow H^{(L)}
     $$

   ##### Fuzzy Rule and Fusion Pass
   - Initialize final embedding matrix:
     $$
     H' \in \mathbb{R}^{|V| \times d}
     $$

   - **For each node $u \in V$:**
     
     - Compute topological features:
       - Degree $d(u)$
       - Clustering coefficient $C(u)$
       - 2-hop label agreement $L(u)$

     - Construct fact vector:
       $$
       \mathbf{f}_u \leftarrow [d(u), C(u), L(u)]
       $$

     - **For $i = 1$ to $3$:**
       $$
       r_i(u) \leftarrow \sigma\big(\alpha_i (f_u[i] - \theta_i)\big)
       $$

     - Form rule activation vector:
       $$
       \mathbf{r}(u) \leftarrow [r_1(u), r_2(u), r_3(u)]
       $$

     - Extract GNN embedding:
       $$
       \mathbf{h}_u \leftarrow H[u, :]
       $$

     - Project rule vector:
       $$
       \mathbf{e}_u \leftarrow W_r \mathbf{r}(u) + b_r
       $$

     - Compute gate:
       $$
       \mathbf{g}_u \leftarrow \sigma\big(W_g [\mathbf{h}_u \,\|\, \mathbf{e}_u] + b_g\big)
       $$

     - Fuse neural and symbolic embeddings:
       $$
       \mathbf{h}'_u \leftarrow \mathbf{g}_u \odot \mathbf{h}_u + (1 - \mathbf{g}_u) \odot \mathbf{e}_u
       $$

     - Store fused embedding:
       $$
       H'[u, :] \leftarrow \mathbf{h}'_u
       $$

   ##### Loss Computation and Backpropagation
   - Compute predictions on labeled nodes:
     $$
     \hat{Y}_{V_L} \leftarrow \text{Classifier}(H'[V_L, :])
     $$

   - Compute loss:
     $$
     \mathcal{L} \leftarrow \text{CrossEntropyLoss}(\hat{Y}_{V_L}, Y_{V_L})
     $$

   - Backpropagate:
     $$
     \mathcal{L}.\text{backward}()
     $$

   - Update all parameters using an optimizer (e.g., Adam).

3. **Return** all trained model parameters.

---